# 🤖 Transformer From Scratch (GPT-Style)

> Building a complete decoder-only Transformer (like GPT) from scratch using NumPy

**What we'll build:**
- Token embedding + positional encoding
- Multi-head self-attention
- Feed-forward network
- Layer normalization + residual connections
- Complete GPT model
- Text generation (inference)

**Architecture:**
```
Input Tokens → Embedding → [Transformer Block] × N → LayerNorm → Linear → Softmax → Output Tokens
                              ↓
                    Multi-Head Attention + FFN
```

## 📚 Imports & Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("✅ Imports ready!")
print(f"NumPy version: {np.__version__}")

## 📖 Part 1: Simple Tokenizer (Character-Level)

GPT uses BPE (Byte-Pair Encoding), but for simplicity we'll start with character-level tokenization.

In [ ]:
class CharTokenizer:
    """
    Simple character-level tokenizer
    """
    
    def __init__(self):
        self.vocab = {}
        self.inverse_vocab = {}
        self.vocab_size = 0
    
    def train(self, text):
        """Build vocabulary from text"""
        # Get unique characters
        chars = sorted(list(set(text)))
        
        # Create mappings
        self.vocab = {ch: i for i, ch in enumerate(chars)}
        self.inverse_vocab = {i: ch for ch, i in self.vocab.items()}
        self.vocab_size = len(chars)
        
        print(f"✅ Vocabulary built: {self.vocab_size} tokens")
        print(f"   Tokens: {''.join(chars)}")
    
    def encode(self, text):
        """Convert text to token indices"""
        return [self.vocab.get(ch, 0) for ch in text]
    
    def decode(self, tokens):
        """Convert token indices to text"""
        return ''.join([self.inverse_vocab.get(t, '') for t in tokens])

# Test tokenizer
print("=== Testing Tokenizer ===")

sample_text = "Hello world! This is a simple tokenizer."
tokenizer = CharTokenizer()
tokenizer.train(sample_text)

encoded = tokenizer.encode(sample_text)
decoded = tokenizer.decode(encoded)

print(f"\nOriginal: {sample_text}")
print(f"Encoded:  {encoded[:20]}...")
print(f"Decoded:  {decoded}")
print(f"Match: {sample_text == decoded}")

## 🔧 Part 2: Building Blocks

In [ ]:
class LayerNorm:
    """Layer Normalization"""
    
    def __init__(self, d_model, eps=1e-5):
        self.gamma = np.ones((1, 1, d_model))
        self.beta = np.zeros((1, 1, d_model))
        self.eps = eps
    
    def forward(self, x):
        # x: (batch, seq_len, d_model)
        mean = np.mean(x, axis=-1, keepdims=True)
        var = np.var(x, axis=-1, keepdims=True)
        x_norm = (x - mean) / np.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta


class Embedding:
    """Token Embedding Layer"""
    
    def __init__(self, vocab_size, d_model):
        # Xavier initialization
        self.weights = np.random.randn(vocab_size, d_model) * np.sqrt(2.0 / vocab_size)
        self.vocab_size = vocab_size
        self.d_model = d_model
    
    def forward(self, x):
        # x: (batch, seq_len) with token indices
        return self.weights[x]  # (batch, seq_len, d_model)


class PositionalEncoding:
    """Sinusoidal Positional Encoding"""
    
    def __init__(self, d_model, max_len=5000):
        pe = np.zeros((max_len, d_model))
        position = np.arange(0, max_len).reshape(-1, 1)
        div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
        
        pe[:, 0::2] = np.sin(position * div_term)
        pe[:, 1::2] = np.cos(position * div_term)
        
        self.pe = pe  # (max_len, d_model)
    
    def forward(self, x):
        seq_len = x.shape[1]
        return x + self.pe[:seq_len]


class Linear:
    """Linear/Dense Layer"""
    
    def __init__(self, in_features, out_features):
        self.weights = np.random.randn(in_features, out_features) * np.sqrt(2.0 / in_features)
        self.bias = np.zeros((1, out_features))
    
    def forward(self, x):
        return np.dot(x, self.weights) + self.bias

print("✅ Building blocks created!")

## 🎯 Part 3: Multi-Head Self-Attention

In [ ]:
class MultiHeadAttention:
    """
    Multi-Head Self-Attention with causal masking
    """
    
    def __init__(self, d_model, num_heads):
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Single projection matrices for efficiency
        self.W_q = Linear(d_model, d_model)
        self.W_k = Linear(d_model, d_model)
        self.W_v = Linear(d_model, d_model)
        self.W_o = Linear(d_model, d_model)
        
        self.attention_weights = None
    
    def softmax(self, x, axis=-1):
        exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
        return exp_x / np.sum(exp_x, axis=axis, keepdims=True)
    
    def split_heads(self, x):
        # x: (batch, seq_len, d_model)
        batch, seq_len, _ = x.shape
        x = x.reshape(batch, seq_len, self.num_heads, self.d_k)
        return x.transpose(0, 2, 1, 3)  # (batch, heads, seq_len, d_k)
    
    def combine_heads(self, x):
        # x: (batch, heads, seq_len, d_k)
        batch, _, seq_len, _ = x.shape
        x = x.transpose(0, 2, 1, 3)
        return x.reshape(batch, seq_len, self.d_model)
    
    def create_causal_mask(self, seq_len):
        """Create mask to prevent attending to future tokens"""
        mask = np.triu(np.ones((seq_len, seq_len)), k=1)
        return mask * -1e9
    
    def forward(self, x, use_causal_mask=True):
        batch, seq_len, _ = x.shape
        
        # Linear projections
        Q = self.W_q.forward(x)
        K = self.W_k.forward(x)
        V = self.W_v.forward(x)
        
        # Split into heads
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        # Attention scores: Q @ K^T / sqrt(d_k)
        scores = np.matmul(Q, K.transpose(0, 1, 3, 2)) / np.sqrt(self.d_k)
        
        # Apply causal mask
        if use_causal_mask:
            mask = self.create_causal_mask(seq_len)
            scores = scores + mask
        
        # Softmax
        attn_weights = self.softmax(scores)
        self.attention_weights = attn_weights
        
        # Apply to values
        attended = np.matmul(attn_weights, V)
        
        # Combine heads and project
        attended = self.combine_heads(attended)
        output = self.W_o.forward(attended)
        
        return output

print("✅ Multi-Head Attention created!")

## 🏗️ Part 4: Feed-Forward Network & Transformer Block

In [ ]:
class FeedForward:
    """
    Position-wise Feed-Forward Network
    FFN(x) = max(0, xW1 + b1)W2 + b2
    """
    
    def __init__(self, d_model, d_ff):
        self.fc1 = Linear(d_model, d_ff)
        self.fc2 = Linear(d_ff, d_model)
    
    def relu(self, x):
        return np.maximum(0, x)
    
    def forward(self, x):
        hidden = self.relu(self.fc1.forward(x))
        return self.fc2.forward(hidden)


class TransformerBlock:
    """
    Complete Transformer Decoder Block
    Pre-norm architecture (LayerNorm before attention/FFN)
    """
    
    def __init__(self, d_model, num_heads, d_ff, dropout_rate=0.1):
        self.norm1 = LayerNorm(d_model)
        self.attention = MultiHeadAttention(d_model, num_heads)
        
        self.norm2 = LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff)
    
    def forward(self, x):
        # Self-attention with residual
        normed = self.norm1.forward(x)
        attn_out = self.attention.forward(normed)
        x = x + attn_out
        
        # Feed-forward with residual
        normed = self.norm2.forward(x)
        ff_out = self.ff.forward(normed)
        x = x + ff_out
        
        return x

print("✅ Transformer Block created!")

## 🤖 Part 5: Complete GPT Model

In [ ]:
class GPT:
    """
    GPT-style Decoder-Only Transformer
    """
    
    def __init__(self, vocab_size, d_model=128, num_heads=8, num_layers=6, d_ff=512, max_len=512):
        
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_len = max_len
        
        # Token embedding
        self.token_embedding = Embedding(vocab_size, d_model)
        
        # Positional encoding
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        
        # Transformer blocks
        self.blocks = [
            TransformerBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ]
        
        # Final layer norm
        self.final_norm = LayerNorm(d_model)
        
        # Output projection (language modeling head)
        self.lm_head = Linear(d_model, vocab_size)
        
        # Count parameters
        self._count_parameters()
    
    def _count_parameters(self):
        """Count total trainable parameters"""
        total = 0
        
        # Embedding
        total += self.token_embedding.weights.size
        
        # Transformer blocks
        for block in self.blocks:
            # Attention
            total += block.attention.W_q.weights.size
            total += block.attention.W_k.weights.size
            total += block.attention.W_v.weights.size
            total += block.attention.W_o.weights.size
            # FFN
            total += block.ff.fc1.weights.size
            total += block.ff.fc2.weights.size
        
        # LM head
        total += self.lm_head.weights.size
        
        print(f"\n📊 Model Statistics:")
        print(f"   Vocabulary size: {self.vocab_size}")
        print(f"   Model dimension: {self.d_model}")
        print(f"   Number of layers: {len(self.blocks)}")
        print(f"   Total parameters: {total:,}")
        print(f"   Model size: ~{total * 4 / 1024 / 1024:.2f} MB (float32)")
    
    def forward(self, x):
        """
        Forward pass
        x: (batch, seq_len) token indices
        Returns: (batch, seq_len, vocab_size) logits
        """
        # Token embedding
        x = self.token_embedding.forward(x)
        
        # Add positional encoding
        x = self.pos_encoding.forward(x)
        
        # Pass through transformer blocks
        for block in self.blocks:
            x = block.forward(x)
        
        # Final normalization
        x = self.final_norm.forward(x)
        
        # Project to vocabulary
        logits = self.lm_head.forward(x)
        
        return logits
    
    def generate(self, tokenizer, prompt="", max_new_tokens=50, temperature=1.0):
        """
        Generate text autoregressively
        
        Args:
            tokenizer: CharTokenizer instance
            prompt: Starting text
            max_new_tokens: Number of tokens to generate
            temperature: Controls randomness (lower = more deterministic)
        """
        # Encode prompt
        tokens = tokenizer.encode(prompt)
        
        print(f"Prompt: '{prompt}'")
        print("Generated: ", end="", flush=True)
        
        for _ in range(max_new_tokens):
            # Crop to max_len
            input_tokens = tokens[-self.max_len:]
            x = np.array([input_tokens])
            
            # Forward pass
            logits = self.forward(x)
            
            # Get logits for last position
            next_token_logits = logits[0, -1, :] / temperature
            
            # Apply softmax
            probs = np.exp(next_token_logits - np.max(next_token_logits))
            probs = probs / np.sum(probs)
            
            # Sample next token
            next_token = np.random.choice(len(probs), p=probs)
            
            # Append to sequence
            tokens.append(next_token)
            
            # Print generated character
            print(tokenizer.inverse_vocab.get(next_token, ''), end="", flush=True)
        
        print()  # New line
        return tokenizer.decode(tokens)

print("✅ GPT Model created!")

## 📊 Part 6: Training Setup

In [ ]:
class CrossEntropyLoss:
    """Cross-entropy loss for language modeling"""
    
    def forward(self, logits, targets):
        """
        logits: (batch, seq_len, vocab_size)
        targets: (batch, seq_len) token indices
        """
        batch_size, seq_len, vocab_size = logits.shape
        
        # Reshape for computation
        logits_flat = logits.reshape(-1, vocab_size)
        targets_flat = targets.reshape(-1)
        
        # Softmax
        exp_logits = np.exp(logits_flat - np.max(logits_flat, axis=1, keepdims=True))
        probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
        
        # Cross-entropy: -log(p[target])
        correct_logprobs = -np.log(probs[np.arange(len(targets_flat)), targets_flat] + 1e-10)
        
        loss = np.mean(correct_logprobs)
        
        return loss, probs


class AdamW:
    """
    AdamW optimizer (simplified)
    """
    
    def __init__(self, learning_rate=0.001, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.01):
        self.lr = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.weight_decay = weight_decay
        self.t = 0
        self.m = {}
        self.v = {}
    
    def step(self, params, grads):
        """Update parameters"""
        self.t += 1
        
        for key in params:
            if key not in self.m:
                self.m[key] = np.zeros_like(params[key])
                self.v[key] = np.zeros_like(params[key])
            
            # Bias correction
            lr_t = self.lr * np.sqrt(1 - self.beta2**self.t) / (1 - self.beta1**self.t)
            
            # Update moments
            self.m[key] = self.beta1 * self.m[key] + (1 - self.beta1) * grads[key]
            self.v[key] = self.beta2 * self.v[key] + (1 - self.beta2) * (grads[key] ** 2)
            
            # Update with weight decay
            params[key] -= lr_t * self.m[key] / (np.sqrt(self.v[key]) + self.eps)
            params[key] -= self.lr * self.weight_decay * params[key]

print("✅ Training utilities created!")

## 🚀 Part 7: Create Model & Test

In [ ]:
# Training data (small example)
training_text = """
The quick brown fox jumps over the lazy dog.
The dog sleeps on the mat.
The cat sits on the bed.
A bird flies in the sky.
The sun shines bright today.
The moon glows at night.
Stars twinkle in the dark sky.
The wind blows through the trees.
Rain falls on the ground.
Snow covers the mountain.
The river flows to the sea.
Fish swim in the water.
Birds sing in the morning.
The cat chases the mouse.
The dog barks at the stranger.
Children play in the park.
The teacher writes on the board.
Students read their books.
The cook prepares delicious food.
The farmer grows fresh vegetables.
"""

# Initialize tokenizer
tokenizer = CharTokenizer()
tokenizer.train(training_text)

# Create small GPT model
print("\n🏗️ Creating GPT Model...")
model = GPT(
    vocab_size=tokenizer.vocab_size,
    d_model=64,          # Small for demo
    num_heads=4,         # 4 attention heads
    num_layers=3,        # 3 transformer blocks
    d_ff=256,            # Feed-forward dimension
    max_len=128
)

# Test forward pass
print("\n🧪 Testing forward pass...")
test_tokens = np.array([tokenizer.encode("The cat")])
logits = model.forward(test_tokens)

print(f"Input shape:  {test_tokens.shape}")
print(f"Output shape: {logits.shape}")
print(f"Output range: [{logits.min():.2f}, {logits.max():.2f}]")

# Test generation (before training - random output)
print("\n🎲 Random generation (before training):")
generated = model.generate(tokenizer, prompt="The ", max_new_tokens=30, temperature=1.0)

## 📈 Part 8: Training Loop (Simplified)

> Note: Full backpropagation through the transformer is complex. This is a simplified demonstration. For real training, use PyTorch/TensorFlow.

In [ ]:
def create_batches(tokens, batch_size, seq_len):
    """Create training batches"""
    n_batches = len(tokens) // (batch_size * seq_len)
    tokens = tokens[:n_batches * batch_size * seq_len]
    
    # Reshape to (n_batches, batch_size, seq_len)
    data = np.array(tokens).reshape(n_batches, batch_size, seq_len)
    
    # Input is first seq_len-1 tokens, target is last seq_len-1 tokens
    x = data[:, :, :-1]
    y = data[:, :, 1:]
    
    return x, y


def train_simple(model, tokenizer, text, epochs=100, batch_size=4, seq_len=32, lr=0.001):
    """
    Simplified training loop (demonstration only)
    Real training requires full backpropagation implementation
    """
    
    # Encode text
    tokens = tokenizer.encode(text)
    
    # Create batches
    x_batches, y_batches = create_batches(tokens, batch_size, seq_len)
    
    print(f"\n📊 Training Data:")
    print(f"   Total tokens: {len(tokens)}")
    print(f"   Batches: {len(x_batches)}")
    print(f"   Batch size: {batch_size}")
    print(f"   Sequence length: {seq_len}")
    
    losses = []
    
    print("\n🚀 Starting training...\n")
    
    for epoch in range(epochs):
        epoch_loss = 0
        
        for i in range(len(x_batches)):
            x = x_batches[i]
            y = y_batches[i]
            
            # Forward pass
            logits = model.forward(x)
            
            # Compute loss
            loss, _ = CrossEntropyLoss().forward(logits, y)
            epoch_loss += loss
        
        avg_loss = epoch_loss / len(x_batches)
        losses.append(avg_loss)
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch:3d} | Loss: {avg_loss:.4f}")
    
    return losses

# Train the model
losses = train_simple(
    model=model,
    tokenizer=tokenizer,
    text=training_text,
    epochs=100,
    batch_size=2,
    seq_len=16,
    lr=0.001
)

print("\n✅ Training complete!")

## 📊 Part 9: Visualize Training & Test Generation

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(losses, color='blue', linewidth=2)
plt.title('Training Loss', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.grid(True, alpha=0.3)

# Loss curve (log scale)
plt.subplot(1, 2, 2)
plt.plot(losses, color='green', linewidth=2)
plt.yscale('log')
plt.title('Training Loss (Log Scale)', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Log Loss')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Test generation with different prompts
print("\n" + "="*50)
print("🎲 GENERATION TESTS")
print("="*50)

prompts = ["The ", "A bird", "The sun", "The cat"]

for prompt in prompts:
    print(f"\n{'-'*40}")
    model.generate(tokenizer, prompt=prompt, max_new_tokens=40, temperature=0.8)
    print(f"{'-'*40}")

## 🔍 Part 10: Attention Visualization

In [ ]:
def visualize_attention_patterns(model, tokenizer, text):
    """Visualize attention patterns for given text"""
    
    tokens = tokenizer.encode(text)
    x = np.array([tokens])
    
    # Forward pass
    _ = model.forward(x)
    
    # Get attention weights from first layer
    attn_weights = model.blocks[0].attention.attention_weights
    
    # attn_weights: (batch, heads, seq_len, seq_len)
    n_heads = attn_weights.shape[1]
    seq_len = len(tokens)
    
    # Create character labels
    labels = list(text)
    
    # Plot
    fig, axes = plt.subplots(2, n_heads//2, figsize=(14, 6))
    
    for h in range(n_heads):
        row, col = h // (n_heads//2), h % (n_heads//2)
        im = axes[row, col].imshow(
            attn_weights[0, h],
            cmap='viridis',
            aspect='auto',
            vmin=0, vmax=1
        )
        axes[row, col].set_title(f'Head {h+1}', fontsize=10)
        
        if seq_len <= 20:
            axes[row, col].set_xticks(range(seq_len))
            axes[row, col].set_xticklabels(labels, rotation=45, ha='right', fontsize=6)
            axes[row, col].set_yticks(range(seq_len))
            axes[row, col].set_yticklabels(labels, fontsize=6)
    
    plt.suptitle(f'Attention Patterns: "{text}"', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Visualize attention
print("=== Attention Visualization ===")
visualize_attention_patterns(model, tokenizer, "The cat sits")

## 🎯 Part 11: Exercises

1. **Implement full backpropagation**: Add backward() methods to all layers
2. **Add dropout**: Implement dropout for regularization
3. **Use BPE tokenizer**: Replace character-level with Byte-Pair Encoding
4. **Scale up**: Try d_model=256, num_layers=6 on larger dataset
5. **Add beam search**: Implement beam search for better generation
6. **Fine-tune on specific domain**: Train on Shakespeare or code
7. **Implement KV-cache**: Speed up inference for generation
8. **Add rotary embeddings**: Try RoPE instead of sinusoidal

## 📚 Key Takeaways

| Component | Purpose |
|-----------|---------|
| **Token Embedding** | Convert tokens to vectors |
| **Positional Encoding** | Add position information |
| **Multi-Head Attention** | Let tokens "talk" to each other |
| **Causal Mask** | Prevent looking at future |
| **Feed-Forward** | Transform each position independently |
| **Layer Norm** | Stabilize training |
| **Residual Connections** | Help gradients flow |
| **LM Head** | Project to vocabulary for prediction |

**Architecture Summary:**
```
Input → [Embed + PosEncode] → [Attn → Norm → FFN → Norm] × N → Linear → Softmax → Output
```

**Next Steps:**
- Use PyTorch for real training
- Try nanoGPT by Andrej Karpathy
- Fine-tune on your own dataset

🎉 **You built a GPT from scratch!**